In [ ]:
%load_ext autoreload
%autoreload 2

import os
import json
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

from plotnine import *
import matplotlib.pyplot as plt

## BRCA1

### Findlay 2018

Comeback to this. It is hg19 :(

In [ ]:
b18f = pl.read_excel(
    "/s/project/deeprvat/ukb_gym/experimental_assays/brca1/BRCA1_Findlay2018.xlsx", 
    sheet_name="Sheet1", 
    read_options={"skip_rows": 2},
    has_header=False
)

# Step 1: extract first row
new_header = b18f.row(0)  # returns a tuple

# Step 2: assign as new header and drop first row
b18f = b18f.slice(1).rename({old: new for old, new in zip(b18f.columns, new_header)})
# b18f.write_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/brca1/BRCA1_Findlay2018.parquet')
b18f

### Dace 2025

In [ ]:
b25d_common = pl.read_excel(
    "/s/project/deeprvat/ukb_gym/experimental_assays/brca1/BRCA1_Dace2025.xlsx", 
    sheet_name="ST3", 
    read_options={"skip_rows": 3},
    has_header=False
)

# Step 1: extract first row
new_header = b25d_common.row(0)  # returns a tuple

# Step 2: assign as new header and drop first row
b25d_common = b25d_common.slice(1).rename({old: new for old, new in zip(b25d_common.columns, new_header)})
b25d_common.write_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/brca1/BRCA1_Dace2025_common.parquet')
b25d_common

In [ ]:
b25d_hap1 = pl.read_excel(
    "/s/project/deeprvat/ukb_gym/experimental_assays/brca1/BRCA1_Dace2025.xlsx", 
    sheet_name="ST1", 
    read_options={"skip_rows": 3},
    has_header=False
)

# Step 1: extract first row
new_header = b25d_hap1.row(0)  # returns a tuple

# Step 2: assign as new header and drop first row
b25d_hap1 = b25d_hap1.slice(1).rename({old: new for old, new in zip(b25d_hap1.columns, new_header)})
b25d_hap1.write_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/brca1/BRCA1_Dace2025_HAP1.parquet')
b25d_hap1

In [ ]:
b25d_hmec = pl.read_excel(
    "/s/project/deeprvat/ukb_gym/experimental_assays/brca1/BRCA1_Dace2025.xlsx", 
    sheet_name="ST4", 
    read_options={"skip_rows": 3},
    has_header=False
)

# Step 1: extract first row
new_header = b25d_hmec.row(0)  # returns a tuple

# Step 2: assign as new header and drop first row
b25d_hmec = b25d_hmec.slice(1).rename({old: new for old, new in zip(b25d_hmec.columns, new_header)})
b25d_hmec.write_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/brca1/BRCA1_Dace2025_HMEC.parquet')
b25d_hmec

## BRCA2

### Sahu 2025

In [ ]:
b25s = pl.read_excel(
    "/s/project/deeprvat/ukb_gym/experimental_assays/brca2/BRCA2_Sahu2025.xlsx", 
    sheet_name="Sheet 1"
)
b25s

In [ ]:
pattern = r"(\d+)([A-Z]+)>([A-Z]+)"

b25s_df = b25s.with_columns(
    pl.lit('BRCA2_Sahu2025_SGE_EScell').alias('source'),
    pl.lit('BRCA2_Sahu2025_SGE_EScell').alias('assay_description'),
    pl.lit('SGE_EScell').alias('assay_name'),
    pl.lit('BRCA2').alias('gene_name'),
    pl.lit('ENSG00000139618').alias('region'),
    pl.lit('chr13').alias('chrom'),
    pl.col('g.nom').str.split('.').list.get(2).str.extract_groups(pattern).alias("parsed_variant")
).unnest(
    # Step 2: Expand the struct fields into new columns
    "parsed_variant"
).rename(
    # Step 3: Rename the new columns to what you want.
    { "1": "pos", "2": "ref", "3": "alt", "AA.change": "mutant", "Function.score": "score"}
).with_columns(
    pl.col('pos').cast(pl.Int64),
    (pl.col('chrom') + ':' + pl.col('pos').cast(pl.Utf8) + ':' + pl.col('ref') + ':' + pl.col('alt')).alias('id'),
    (pl.col('assay_name') + '_' + pl.col('source') + '_' + pl.col('assay_description').str.replace(" ", "_")).alias('assay_id'),
).select(
    ['source', 'assay_description', 'assay_id', 'assay_name', 'gene_name', 'region', 'id', 'mutant', 'score']
).drop_nulls()

b25s_df

In [ ]:
b25s_df.write_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/brca2/BRCA2_SGE_Sahu2025.parquet')

### Huang 2025

In [ ]:
b25h = pl.read_excel(
    "/s/project/deeprvat/ukb_gym/experimental_assays/brca2/BRCA2_Huang2025.xlsx", 
    sheet_name="Table S3",
    read_options={"skip_rows": 1},
    has_header=False
)

new_header = b25h.row(0)  

b25h = b25h.slice(1).rename({old: new for old, new in zip(b25h.columns, new_header)})
b25h

In [ ]:
b25h_df = b25h.with_columns(
    pl.lit('BRCA2_Huang2025_SGE_HAP1').alias('source'),
    pl.lit('BRCA2_Sahu2025_SGE_HAP1').alias('assay_description'),
    pl.lit('SGE_HAP1').alias('assay_name'),
    pl.lit('BRCA2').alias('gene_name'),
    pl.lit('ENSG00000139618').alias('region'),
    pl.lit('chr13').alias('chrom'),
).rename({
    "GRCh38Location": "pos", 
    "REF": "ref", 
    "ALT": "alt", 
    "Amino acid change (p.)": "mutant", 
    "Model based functional score": "score",
    "Posterior probability of pathogenicity": "post_prob_patho"
}).with_columns(
    pl.col('pos').cast(pl.Int64),
    pl.col('score').cast(pl.Float64),
    pl.col('post_prob_patho').cast(pl.Float64),
    (pl.col('chrom') + ':' + pl.col('pos').cast(pl.Utf8) + ':' + pl.col('ref') + ':' + pl.col('alt')).alias('id'),
    (pl.col('assay_name') + '_' + pl.col('source') + '_' + pl.col('assay_description').str.replace(" ", "_")).alias('assay_id'),
).select(
    ['source', 'assay_description', 'assay_id', 'assay_name', 'gene_name', 'region', 'id', 'mutant', 'score']
).drop_nulls()

b25h_df

In [ ]:
b25h_df.write_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/brca2/BRCA2_SGE_Huang2025.parquet')

In [ ]:
b25h_df[['id', 'score']].join(b25s_df[['id', 'score']], on='id', how='inner').rename({'score': 'score_huang', 'score_right': 'score_sahu'})